
# CELL 1
#### Mount Drive + create full CareSense (my component) folder structure
####Run this once per session. Safe to re-run any time (idempotent).
#

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/CareSense'

FOLDERS = [
    f'{BASE}/data/raw',         # original .csv.gz files from FTP — private, never share
    f'{BASE}/data/processed',   # cleaned/derived dataframes (parquet)
    f'{BASE}/data/cache',       # intermediate checkpoints (value counts, joins, summaries)
    f'{BASE}/notebooks',        # saved copies of notebooks, if you want manual backups
    f'{BASE}/docs',             # READMEs, audit notes, decision logs
]

for folder in FOLDERS:
    os.makedirs(folder, exist_ok=True)
    print(f'OK: {folder}')

# Convenience path variables for use in later cells
RAW   = f'{BASE}/data/raw'
PROC  = f'{BASE}/data/processed'
CACHE = f'{BASE}/data/cache'
DOCS  = f'{BASE}/docs'

print('\nFolder structure ready:')
for folder in FOLDERS:
    print(' -', folder)

Mounted at /content/drive
OK: /content/drive/MyDrive/CareSense/data/raw
OK: /content/drive/MyDrive/CareSense/data/processed
OK: /content/drive/MyDrive/CareSense/data/cache
OK: /content/drive/MyDrive/CareSense/notebooks
OK: /content/drive/MyDrive/CareSense/docs

Folder structure ready:
 - /content/drive/MyDrive/CareSense/data/raw
 - /content/drive/MyDrive/CareSense/data/processed
 - /content/drive/MyDrive/CareSense/data/cache
 - /content/drive/MyDrive/CareSense/notebooks
 - /content/drive/MyDrive/CareSense/docs


# Cell 02
#### Install and import every package this notebook needs, all in one place
#### pandas is used for loading and analyzing the survey/sensor data
#### pyarrow lets pandas save and load fast .parquet cache files
#### paramiko is available in case we need SFTP instead of FTP later
#### ftplib is used to connect to the TILES FTP+TLS server
#### getpass is used so the FTP password is typed in, never saved in the notebook

In [2]:
!pip install -q pandas pyarrow paramiko

import os
import ftplib
import pandas as pd
from getpass import getpass

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.9/208.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.1 MB/s eta 0:00:00


#Cell 03
#### Connecting to FTP Server

In [3]:
from getpass import getpass
FTP_PASS = getpass('FTP password: ')

def connect_ftp():
    ftp = ftplib.FTP_TLS('tiles-data.isi.edu')
    ftp.login('Dayana_Kumar', FTP_PASS)
    ftp.prot_p()
    return ftp

FTP password: ··········


# Cell 04
#### Defining a reusable function to download files from the TILES FTP server
#### Checks whether the file already exists in the Drive raw folder first
#### Skips downloading and reuses the existing file if it is already present
#### Connects to the FTP server only when the file does not exist locally
#### Saves the downloaded file into the Drive raw data folder
#### Returns the local file path so later cells can load it directly

In [4]:
def download_if_missing(remote_path, local_filename):
    local_path = f'{RAW}/{local_filename}'
    if os.path.exists(local_path):
        print(f'Already have {local_filename}, skipping download.')
        return local_path
    ftp = connect_ftp()
    with open(local_path, 'wb') as f:
        ftp.retrbinary(f'RETR {remote_path}', f.write)
    ftp.quit()
    print(f'Downloaded {local_filename}')
    return local_path

# Cell 05 — SUPERSEDED, kept for research record only
#### This downloaded the wrong file (timings only, no actual answers)
#### Do not use ema_path for analysis — see Cell 10 for the correct file

In [5]:
ema_path = download_if_missing(
    '/surveys/raw/EMAs/job_personality_health-context_timings.csv.gz',
    'ema_job_personality_health.csv.gz'
)

Already have ema_job_personality_health.csv.gz, skipping download.


# Cell 06
#### Define a helper function to print clearly labeled section headers
#### Used before every analysis output so results are easy to read
#### Keeps a consistent, professional format across all cells going forward
#### Also increases pandas display limits so outputs are not truncated

In [6]:
def section(title):
    print('=' * 60)
    print(title)
    print('=' * 60)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

# Cell 07
#### Load the downloaded EMA file into a pandas dataframe
#### Confirms the file loads correctly and shows its basic shape
#### Shows the column names so we can confirm the stress field exists
#### This is the first look at real data, not just documentation

In [7]:
section('EMA File — Load and Basic Shape')

df_ema = pd.read_csv(ema_path, compression='gzip')

print(f'Rows: {len(df_ema)}')
print(f'Columns: {len(df_ema.columns)}')
print('\nColumn names:')
print(list(df_ema.columns))

EMA File — Load and Basic Shape
Rows: 14795
Columns: 86

Column names:
['participant_id', 'survey_type', 'sent_ts', 'start_ts', 'completed_ts', 'duration', 'has_finished', 'context1', 'Context1Time_1', 'Context1Time_2', 'Context1Time_3', 'Context1Time_4', 'context2', 'context2_TEXT', 'Context2Time_1', 'Context2Time_2', 'Context2Time_3', 'Context2Time_4', 'context3', 'context3_TEXT', 'Context3Time_1', 'Context3Time_2', 'Context3Time_3', 'Context3Time_4', 'context4', 'context4_TEXT', 'Context4Time_1', 'Context4Time_2', 'Context4Time_3', 'Context4Time_4', 'StressTime_1', 'StressTime_2', 'StressTime_3', 'StressTime_4', 'AnxietyTime_1', 'AnxietyTime_2', 'AnxietyTime_3', 'AnxietyTime_4', 'PANASTime_1', 'PANASTime_2', 'PANASTime_3', 'PANASTime_4', 'bfidTime_1', 'bfidTime_2', 'bfidTime_3', 'bfidTime_4', 'sleepTime_1', 'sleepTime_2', 'sleepTime_3', 'sleepTime_4', 'exTime_1', 'exTime_2', 'exTime_3', 'exTime_4', 'tob1Time_1', 'tob1Time_2', 'tob1Time_3', 'tob1Time_4', 'tob2Time_1', 'tob2Time_2', '

# Cell 08
#### Check how many unique participants have EMA responses
#### Calculate how many times each participant answered the survey
#### Show summary statistics (min, max, mean, median responses per person)
#### This tells us whether the data has enough temporal density to use

In [8]:
section('EMA Responses per Participant')

unique_participants = df_ema['participant_id'].nunique()
print(f'Unique participants with EMA data: {unique_participants}')

responses_per_participant = df_ema.groupby('participant_id').size()

section('Responses per Participant — Summary Statistics')
print(responses_per_participant.describe())

EMA Responses per Participant
Unique participants with EMA data: 212
Responses per Participant — Summary Statistics
count    212.000000
mean      69.787736
std        6.577085
min       15.000000
25%       70.000000
50%       71.000000
75%       71.000000
max       72.000000
dtype: float64


# Cell 09
#### List the exact filenames inside the surveys EMAs folder on FTP
#### We downloaded the timings-only file by mistake, not the actual responses
#### This confirms the exact name of the file containing real answer values
#### Avoids guessing or mistyping a filename that could fail silently

In [9]:
section('Listing exact filenames in /surveys/raw/EMAs')

ftp = connect_ftp()
ftp.cwd('/surveys/raw/EMAs')
filenames = ftp.nlst()
ftp.quit()

for name in filenames:
    print(name)

Listing exact filenames in /surveys/raw/EMAs
README.md
job_personality_health-context_stress_anxiety_pand_bfid_sleep_ex_tob_alc_work_itpd_irbd_dalal.csv.gz
job_personality_health-context_timings.csv.gz
psychological_capital-Location_Activity.csv.gz
psychological_capital-Psycap_Location_Activity_Engage_IS_CS_HS.csv.gz
psychological_flexibility-Activity.csv.gz
psychological_flexibility-Activity_Experience_PF.csv.gz


# Cell 10
#### Download the CORRECT Job, Personality, and Health EMA responses file
#### This file contains the actual stress, anxiety, and other answer values
#### Different from the timings-only file downloaded previously by mistake
#### Uses the same download_if_missing function, safe to rerun
#### Saves as ema_job_personality_health_responses.csv.gz

In [10]:
ema_responses_path = download_if_missing(
    '/surveys/raw/EMAs/job_personality_health-context_stress_anxiety_pand_bfid_sleep_ex_tob_alc_work_itpd_irbd_dalal.csv.gz',
    'ema_job_personality_health_responses.csv.gz'
)

Already have ema_job_personality_health_responses.csv.gz, skipping download.


# Cell 11
#### Load the correct EMA responses file into a new dataframe
#### Confirm the file loads correctly and show its shape
#### Show the column names to confirm the stress column now actually exists

In [11]:
section('EMA Responses File — Load and Basic Shape')

df_ema_resp = pd.read_csv(ema_responses_path, compression='gzip')

print(f'Rows: {len(df_ema_resp)}')
print(f'Columns: {len(df_ema_resp.columns)}')
print('\nColumn names:')
print(list(df_ema_resp.columns))

EMA Responses File — Load and Basic Shape
Rows: 14795
Columns: 150

Column names:
['participant_id', 'survey_type', 'sent_ts', 'start_ts', 'completed_ts', 'duration', 'has_finished', 'context1', 'Context1Time_1', 'Context1Time_2', 'Context1Time_3', 'Context1Time_4', 'context2', 'context2_TEXT', 'Context2Time_1', 'Context2Time_2', 'Context2Time_3', 'Context2Time_4', 'context3', 'context3_TEXT', 'Context3Time_1', 'Context3Time_2', 'Context3Time_3', 'Context3Time_4', 'context4', 'context4_TEXT', 'Context4Time_1', 'Context4Time_2', 'Context4Time_3', 'Context4Time_4', 'stress', 'StressTime_1', 'StressTime_2', 'StressTime_3', 'StressTime_4', 'anxiety', 'AnxietyTime_1', 'AnxietyTime_2', 'AnxietyTime_3', 'AnxietyTime_4', 'pand1', 'pand2', 'pand3', 'pand4', 'pand5', 'pand6', 'pand7', 'pand8', 'pand9', 'pand10', 'PANASTime_1', 'PANASTime_2', 'PANASTime_3', 'PANASTime_4', 'bfid1', 'bfid2', 'bfid3', 'bfid4', 'bfid5', 'bfid6', 'bfid7', 'bfid8', 'bfid9', 'bfid10', 'bfidTime_1', 'bfidTime_2', 'bfidTi

# Cell 12
#### Check how many unique participants have EMA responses in this file
#### Calculate how many times each participant answered the survey
#### Show summary statistics on responses per participant

In [12]:
section('EMA Responses per Participant (Correct File)')

unique_participants = df_ema_resp['participant_id'].nunique()
print(f'Unique participants with EMA data: {unique_participants}')

responses_per_participant = df_ema_resp.groupby('participant_id').size()

section('Responses per Participant — Summary Statistics')
print(responses_per_participant.describe())

EMA Responses per Participant (Correct File)
Unique participants with EMA data: 212
Responses per Participant — Summary Statistics
count    212.000000
mean      69.787736
std        6.577085
min       15.000000
25%       70.000000
50%       71.000000
75%       71.000000
max       72.000000
dtype: float64


# Cell 13
#### Check how complete the stress field specifically is in this file
#### Count missing values in the stress column
#### Calculate the percentage of missing stress responses
#### Show the distribution of stress values (1 to 5 scale)

In [13]:
section('Stress Field — Completeness Check')

total_rows = len(df_ema_resp)
missing_stress = df_ema_resp['stress'].isna().sum()
missing_pct = (missing_stress / total_rows) * 100

print(f'Total rows: {total_rows}')
print(f'Missing stress values: {missing_stress}')
print(f'Missing percentage: {missing_pct:.2f}%')

section('Stress Value Distribution (including missing)')
print(df_ema_resp['stress'].value_counts(dropna=False).sort_index())

Stress Field — Completeness Check
Total rows: 14795
Missing stress values: 3445
Missing percentage: 23.28%
Stress Value Distribution (including missing)
stress
1.0    5702
2.0    2814
3.0    2270
4.0     433
5.0     131
NaN    3445
Name: count, dtype: int64


# Cell 14
#### Check the date range covered by the EMA responses
#### Convert the sent timestamp to a proper datetime format
#### Show the earliest and latest response dates
#### Show a real sample of participant_id, timestamp, and stress values

In [14]:
section('EMA Date Range Coverage')

df_ema_resp['sent_ts_parsed'] = pd.to_datetime(df_ema_resp['sent_ts'], errors='coerce')

print(f'Earliest response: {df_ema_resp["sent_ts_parsed"].min()}')
print(f'Latest response: {df_ema_resp["sent_ts_parsed"].max()}')
print(f'Unparseable timestamps: {df_ema_resp["sent_ts_parsed"].isna().sum()}')

section('Sample Rows (participant_id, sent_ts, stress)')
sample_cols = ['participant_id', 'sent_ts', 'stress']
print(df_ema_resp[sample_cols].dropna(subset=['stress']).head(10).to_string(index=False))

EMA Date Range Coverage
Earliest response: 2018-03-05 06:00:00
Latest response: 2018-07-13 18:00:00
Unparseable timestamps: 0
Sample Rows (participant_id, sent_ts, stress)
                      participant_id             sent_ts  stress
3a62899e-082a-4638-ab8b-b49f1ce12a20 2018-03-05T06:00:00     1.0
c7c4392b-02ee-40f4-a22f-ab2386c6ccf5 2018-03-05T06:00:00     2.0
7e6a98f1-73c3-4023-9bbd-acf2b70d04b5 2018-03-05T06:00:00     1.0
8b13d979-315f-4357-8f0e-7c12df0a6ca8 2018-03-05T06:00:00     3.0
8307ff6e-f582-49b0-81cc-39fe9966d097 2018-03-05T06:00:00     1.0
77dfe9b8-5f40-49a6-ba09-706198bb8a48 2018-03-05T06:00:00     1.0
efc0dabd-1c17-4a63-89ca-21f65250e43f 2018-03-05T06:00:00     2.0
6ceb4ef3-6578-45cc-bc02-fa97614313e1 2018-03-05T06:00:00     3.0
df3a6b7a-7e27-4003-beea-2cb5cf08da83 2018-03-05T06:00:00     3.0
a9dfbe4d-4076-48c7-a72b-342fe4c12514 2018-03-05T06:00:00     2.0


# Cell 15
#### Check whether stress missingness is spread across participants
#### or concentrated in a few participants who barely responded
#### This tells us if 23% missing is a general pattern or a few outliers

In [15]:
section('Stress Missingness per Participant')

missing_by_participant = df_ema_resp.groupby('participant_id')['stress'].apply(
    lambda x: x.isna().mean() * 100
)

section('Missing % per Participant — Summary Statistics')
print(missing_by_participant.describe())

section('Participants with over 50% missing stress responses')
high_missing = missing_by_participant[missing_by_participant > 50]
print(f'Count: {len(high_missing)}')
print(high_missing.sort_values(ascending=False).head(10))

Stress Missingness per Participant
Missing % per Participant — Summary Statistics
count    212.000000
mean      24.208932
std       24.907142
min        0.000000
25%        5.714286
50%       14.084507
75%       32.510060
max      100.000000
Name: stress, dtype: float64
Participants with over 50% missing stress responses
Count: 30
participant_id
24166136-6ee3-4521-abaf-972fbe83d15d    100.000000
900bd2b1-a14f-4775-8f78-07ba77caf23b    100.000000
f30085f9-fdb0-49fb-b242-2ad8286b242c    100.000000
803be457-cee4-4f3f-9540-ff574c57c697     97.183099
1b1bcdb5-00eb-4a8c-8c05-a70d678ca0f8     95.714286
fb3a4dd7-3cb5-438e-a75d-244d1aec4790     94.285714
94144e0e-7ea9-4b65-96a4-86e9741d269d     93.333333
be0e8360-24c6-4d74-ab72-aadf8d8acc82     90.140845
41f2fcc5-e1a6-42d0-9b51-68682d173e6f     87.142857
91297102-4775-4ea1-a10d-4530e3c2f0af     85.714286
Name: stress, dtype: float64


# Cell 16
#### Check how many distinct calendar days each participant has EMA data for
#### This tells us the real temporal spread per person, not just response count
#### Important for personalized baseline and temporal trend modeling

In [16]:
section('Distinct EMA Days per Participant')

df_ema_resp['sent_date'] = df_ema_resp['sent_ts_parsed'].dt.date
days_per_participant = df_ema_resp.groupby('participant_id')['sent_date'].nunique()

print(days_per_participant.describe())

Distinct EMA Days per Participant
count    212.000000
mean      69.627358
std        6.533023
min       15.000000
25%       70.000000
50%       71.000000
75%       71.000000
max       71.000000
Name: sent_date, dtype: float64


# Cell 17
#### Check the survey_type field to see if job/personality/health
#### surveys are separate rows or combined, since the filename suggests
#### this file covers all three survey types together

In [17]:
section('Survey Type Breakdown')

print(df_ema_resp['survey_type'].value_counts(dropna=False))

Survey Type Breakdown
survey_type
health         7278
job            6473
personality    1044
Name: count, dtype: int64


# Cell 18
#### Check the work field to see how many EMA responses occurred
#### on days the participant reported working
#### This matters for aligning EMA stress with workload/physio features later

In [18]:
section('EMA Responses on Work Days')

print(df_ema_resp['work'].value_counts(dropna=False))

EMA Responses on Work Days
work
NaN    9884
2.0    2505
1.0    2406
Name: count, dtype: int64


# Cell 19
#### Save the correct EMA responses dataframe as a cached parquet file
#### This avoids re-downloading or re-loading the raw csv.gz in future sessions
#### Anything downstream should load from this cache, not from ema_responses_path

In [19]:
section('Saving EMA Responses to Cache')

cache_path = f'{CACHE}/ema_job_personality_health_clean.parquet'
df_ema_resp.to_parquet(cache_path)

print(f'Saved to: {cache_path}')
print(f'Rows: {len(df_ema_resp)}, Columns: {len(df_ema_resp.columns)}')

Saving EMA Responses to Cache
Saved to: /content/drive/MyDrive/CareSense/data/cache/ema_job_personality_health_clean.parquet
Rows: 14795, Columns: 152


# Cell 20 (corrected)
#### Lists exact contents one level deeper for fitbit, omsignal, owls/locations
#### Previous run showed these are subfolders, not flat files — this fixes that
#### Also flags that jelly has 206 files vs 212 participants (6 missing)

In [20]:


section('fitbit — listing inside each subfolder')
ftp = connect_ftp()
for sub in ['daily-summary', 'heart-rate', 'sleep', 'sleep-data', 'sleep-metadata', 'step-count', 'sync']:
    try:
        ftp.cwd(f'/fitbit/{sub}')
        items = ftp.nlst()
        print(f'{sub}: {len(items)} items, sample: {items[:3]}')
    except Exception as e:
        print(f'{sub}: ERROR — {e}')
ftp.quit()

section('omsignal — listing inside each subfolder')
ftp = connect_ftp()
for sub in ['ecg', 'features', 'metadata']:
    try:
        ftp.cwd(f'/omsignal/{sub}')
        items = ftp.nlst()
        print(f'{sub}: {len(items)} items, sample: {items[:3]}')
    except Exception as e:
        print(f'{sub}: ERROR — {e}')
ftp.quit()

section('owlinone/owls/locations — listing this path directly')
ftp = connect_ftp()
try:
    ftp.cwd('/owlinone/owls/locations')
    items = ftp.nlst()
    print(f'{len(items)} items, sample: {items[:5]}')
except Exception as e:
    print(f'ERROR — {e}')
    print('Trying one level up instead: /owlinone/owls')
    ftp.cwd('/owlinone/owls')
    print(ftp.nlst())
ftp.quit()

section('FINDING: jelly proximity coverage')
print('jelly folder contains 206 participant files.')
print('Total participants in participant-info: 212.')
print('Gap: 6 participants have no jelly proximity data — needs cross-check '
      'against participant-info IDs once we have both loaded together.')

fitbit — listing inside each subfolder
daily-summary: 209 items, sample: ['02581754-36cd-4b23-85ea-bf995c6dec83.csv.gz', '0271c478-a56a-4c09-ab91-9743184dd71b.csv.gz', '02b7a595-6508-46bd-8239-6deb433d6290.csv.gz']
heart-rate: 209 items, sample: ['02581754-36cd-4b23-85ea-bf995c6dec83.csv.gz', '0271c478-a56a-4c09-ab91-9743184dd71b.csv.gz', '02b7a595-6508-46bd-8239-6deb433d6290.csv.gz']
sleep: 206 items, sample: ['02581754-36cd-4b23-85ea-bf995c6dec83.csv.gz', '0271c478-a56a-4c09-ab91-9743184dd71b.csv.gz', '02b7a595-6508-46bd-8239-6deb433d6290.csv.gz']
sleep-data: 209 items, sample: ['02581754-36cd-4b23-85ea-bf995c6dec83.csv.gz', '0271c478-a56a-4c09-ab91-9743184dd71b.csv.gz', '02b7a595-6508-46bd-8239-6deb433d6290.csv.gz']
sleep-metadata: 209 items, sample: ['02581754-36cd-4b23-85ea-bf995c6dec83.csv.gz', '0271c478-a56a-4c09-ab91-9743184dd71b.csv.gz', '02b7a595-6508-46bd-8239-6deb433d6290.csv.gz']
step-count: 209 items, sample: ['02581754-36cd-4b23-85ea-bf995c6dec83.csv.gz', '0271c478-a56a-

# Cell 21
#### Download README + one real participant's file for fitbit and omsignal
#### Also download the confirmed owls/locations.csv.gz file
#### Uses a known real participant ID already seen in participant-info
#### Prints columns, shape, and a sample for each — fast pass, no guessing

In [21]:


SAMPLE_PID = '02b7a595-6508-46bd-8239-6deb433d6290'

def peek(path, label):
    try:
        df = pd.read_csv(path, compression='gzip')
        print(f'{label}: {df.shape[0]} rows, {df.shape[1]} cols')
        print(f'Columns: {list(df.columns)}')
        print(df.head(3).to_string(index=False))
    except Exception as e:
        print(f'{label}: ERROR — {e}')

# --- READMEs (small, fast) ---
readme_targets = [
    ('/fitbit/heart-rate/README.md', 'fitbit_heartrate_README.md'),
    ('/fitbit/daily-summary/README.md', 'fitbit_dailysummary_README.md'),
    ('/fitbit/sleep/README.md', 'fitbit_sleep_README.md'),
    ('/omsignal/features/README.md', 'omsignal_features_README.md'),
    ('/omsignal/metadata/README.md', 'omsignal_metadata_README.md'),
]

section('Downloading READMEs')
for remote, local in readme_targets:
    try:
        path = download_if_missing(remote, local)
        with open(path, 'r', errors='ignore') as f:
            print(f'\n--- {local} ---')
            print(f.read())
    except Exception as e:
        print(f'{local}: ERROR — {e}')

# --- Real data files for one participant ---
section('fitbit/heart-rate — real file for sample participant')
hr_path = download_if_missing(f'/fitbit/heart-rate/{SAMPLE_PID}.csv.gz', f'hr_{SAMPLE_PID}.csv.gz')
peek(hr_path, 'heart-rate')

section('fitbit/daily-summary — real file for sample participant')
ds_path = download_if_missing(f'/fitbit/daily-summary/{SAMPLE_PID}.csv.gz', f'ds_{SAMPLE_PID}.csv.gz')
peek(ds_path, 'daily-summary')

section('omsignal/features — real file for sample participant')
om_path = download_if_missing(f'/omsignal/features/{SAMPLE_PID}.csv.gz', f'om_{SAMPLE_PID}.csv.gz')
peek(om_path, 'omsignal-features')

section('owlinone/owls/locations.csv.gz — full file (small)')
loc_path = download_if_missing('/owlinone/owls/locations.csv.gz', 'owls_locations.csv.gz')
peek(loc_path, 'owls-locations')

Already have fitbit_heartrate_README.md, skipping download.

--- fitbit_heartrate_README.md ---
This directory contains heart rate measurements from the Fitbit (PPG sensor) per participant and contains the following fields:

 - Timestamp: Date and time of the measurement
 - HeartRatePPG: Measured heart rate in beats per minute

*Caveat* - The timestamps are localized according to each participant's smartphone.  Most of the time, this corresponds to Pacific Time, but the timestamps for some participants may jump or repeat occasionally due to time zone changes.  These timezone changes and daylight saving time adjustment only affect the timestamps after the Fitbit devices have been synchronized to their paired smartphones.

Already have fitbit_dailysummary_README.md, skipping download.

--- fitbit_dailysummary_README.md ---
This directory contains daily summaries of Fitbit data per participant.  The summary data is provided by the Fitbit API and contains the following fields:

*Caveat* - 

# Cell 22 (corrected path)
#### Fix: locations.csv.gz lives inside a 'locations' FOLDER, not at owls/ directly
#### Correct path: /owlinone/owls/locations/locations.csv.gz
#### Also runs omsignal and fitbit heart-rate completeness checks

In [22]:


section('Downloading owls locations.csv.gz — corrected path')

loc_path = download_if_missing(
    '/owlinone/owls/locations/locations.csv.gz',
    'owls_locations.csv.gz'
)
try:
    df_owls_loc = pd.read_csv(loc_path, compression='gzip')
    print(f'Rows: {len(df_owls_loc)}, Columns: {list(df_owls_loc.columns)}')
    print(df_owls_loc.head(10).to_string(index=False))
except Exception as e:
    print(f'Still failing — ERROR: {e}')

section('omsignal/features — real completeness check (sample participant)')

df_om = pd.read_csv(om_path, compression='gzip')
print(f'Total rows: {len(df_om)}')
key_cols = ['HeartRate', 'BreathingRate', 'BreathingDepth', 'AvgHeartRate', 'Cadence', 'Intensity', 'Steps']
for col in key_cols:
    missing_pct = df_om[col].isna().mean() * 100
    print(f'{col}: {missing_pct:.2f}% missing')

section('fitbit/heart-rate — basic quality check (sample participant)')

df_hr = pd.read_csv(hr_path, compression='gzip')
df_hr['Timestamp_parsed'] = pd.to_datetime(df_hr['Timestamp'], errors='coerce')
print(f'Total rows: {len(df_hr)}')
print(f'Date range: {df_hr["Timestamp_parsed"].min()} to {df_hr["Timestamp_parsed"].max()}')
print(f'HeartRatePPG missing: {df_hr["HeartRatePPG"].isna().mean()*100:.2f}%')
print(f'HeartRatePPG range: {df_hr["HeartRatePPG"].min()} to {df_hr["HeartRatePPG"].max()}')

Already have owls_locations.csv.gz, skipping download.
Rows: 243, Columns: ['directory', 'x', 'y']
       directory     x     y
0d7b:lounge:fc51   7.2  -2.7
   0d7b:med:7025   3.0   3.2
    0d7b:ns:938d  -5.6   5.8
    0d7b:ns:a953   6.5  -7.6
   0d7b:pat:0c8b  -7.5  13.2
   0d7b:pat:0d49 -22.8  11.5
   0d7b:pat:2514   0.2  13.2
   0d7b:pat:2783  -6.8  -8.5
   0d7b:pat:4d87 -15.1  11.5
   0d7b:pat:5901   3.5 -20.0
omsignal/features — real completeness check (sample participant)
Total rows: 762192
HeartRate: 13.42% missing
BreathingRate: 68.93% missing
BreathingDepth: 68.93% missing
AvgHeartRate: 99.68% missing
Cadence: 3.26% missing
Intensity: 3.26% missing
Steps: 3.26% missing
fitbit/heart-rate — basic quality check (sample participant)
Total rows: 711776
Date range: 2018-03-05 00:00:03 to 2018-05-14 23:58:46
HeartRatePPG missing: 0.25%
HeartRatePPG range: 36.0 to 180.0


# Cell 23
#### Fix download_if_missing: remove partial/empty file if the FTP transfer fails
#### This prevents future runs from thinking a broken file is already downloaded
#### Then retry the owls locations file with this safer version
#### Then build a first single-participant feature+target prototype table

In [23]:


def download_if_missing(remote_path, local_filename):
    local_path = f'{RAW}/{local_filename}'
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        print(f'Already have {local_filename}, skipping download.')
        return local_path
    try:
        ftp = connect_ftp()
        with open(local_path, 'wb') as f:
            ftp.retrbinary(f'RETR {remote_path}', f.write)
        ftp.quit()
        print(f'Downloaded {local_filename}')
        return local_path
    except Exception as e:
        if os.path.exists(local_path):
            os.remove(local_path)
        print(f'Download failed for {remote_path}: {e}')
        raise

section('Retrying owls locations with fixed function')
loc_path = download_if_missing('/owlinone/owls/locations/locations.csv.gz', 'owls_locations.csv.gz')
df_owls_loc = pd.read_csv(loc_path, compression='gzip')
print(f'Rows: {len(df_owls_loc)}, Columns: {list(df_owls_loc.columns)}')
print(df_owls_loc.head(10).to_string(index=False))

section('PROTOTYPE: Fitbit heart-rate aggregated to daily features, sample participant')

df_hr['date'] = df_hr['Timestamp_parsed'].dt.date
daily_hr = df_hr.groupby('date')['HeartRatePPG'].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
daily_hr.columns = ['date', 'hr_mean', 'hr_std', 'hr_min', 'hr_max', 'hr_sample_count']
print(f'Daily aggregated rows: {len(daily_hr)}')
print(daily_hr.head(10).to_string(index=False))

section('PROTOTYPE: Joining daily physio features to EMA stress for sample participant')

sample_ema = df_ema_resp[df_ema_resp['participant_id'] == SAMPLE_PID].copy()
sample_ema['date'] = sample_ema['sent_ts_parsed'].dt.date
merged_sample = sample_ema[['date', 'stress']].merge(daily_hr, on='date', how='left')
print(f'Merged rows: {len(merged_sample)}')
print(f'Rows with matching physio data: {merged_sample["hr_mean"].notna().sum()}')
print(merged_sample.dropna(subset=['stress']).head(10).to_string(index=False))

Retrying owls locations with fixed function
Already have owls_locations.csv.gz, skipping download.
Rows: 243, Columns: ['directory', 'x', 'y']
       directory     x     y
0d7b:lounge:fc51   7.2  -2.7
   0d7b:med:7025   3.0   3.2
    0d7b:ns:938d  -5.6   5.8
    0d7b:ns:a953   6.5  -7.6
   0d7b:pat:0c8b  -7.5  13.2
   0d7b:pat:0d49 -22.8  11.5
   0d7b:pat:2514   0.2  13.2
   0d7b:pat:2783  -6.8  -8.5
   0d7b:pat:4d87 -15.1  11.5
   0d7b:pat:5901   3.5 -20.0
PROTOTYPE: Fitbit heart-rate aggregated to daily features, sample participant
Daily aggregated rows: 71
      date   hr_mean    hr_std  hr_min  hr_max  hr_sample_count
2018-03-05 79.274316 21.810910    44.0   156.0            14651
2018-03-06 70.441978 17.914310    49.0   168.0            10453
2018-03-07 60.946827 11.697214    41.0   159.0            11077
2018-03-08 57.837455  8.108291    46.0   138.0             9665
2018-03-09 65.695082 20.188008    45.0   174.0            11895
2018-03-10 62.317931 12.023778    44.0   138.0    

# Cell 24 (fixed — self-contained)
#### Lock in the participant-level pool split ONCE — this must never change
#### Now loads participant-info itself instead of depending on an earlier cell
#### Demo pool participants are set aside now and touched by nothing else

In [25]:


import random

section('Loading participant-info (self-contained, no dependency on earlier cells)')

participant_info_path = download_if_missing(
    '/metadata/participant-info/participant-info.csv.gz',
    'participant_info.csv.gz'
)
df_participants = pd.read_csv(participant_info_path, compression='gzip')
print(f'Loaded {len(df_participants)} participants')

section('Locking participant pool split')

pool_file = f'{CACHE}/participant_pools.parquet'

if os.path.exists(pool_file):
    print('Pool split already exists — loading existing split (not regenerating).')
    df_pools = pd.read_parquet(pool_file)
else:
    print('No existing split found — creating it now, ONE TIME ONLY.')
    all_ids = df_participants['ParticipantID'].tolist()

    random.seed(42)  # fixed seed for reproducibility
    shuffled = all_ids.copy()
    random.shuffle(shuffled)

    n_demo = 15
    demo_ids = shuffled[:n_demo]
    remaining_ids = shuffled[n_demo:]

    n_test = 27
    n_val = 30
    test_ids = remaining_ids[:n_test]
    val_ids = remaining_ids[n_test:n_test + n_val]
    train_ids = remaining_ids[n_test + n_val:]

    pool_assignment = []
    for pid in demo_ids:
        pool_assignment.append({'ParticipantID': pid, 'pool': 'demo'})
    for pid in test_ids:
        pool_assignment.append({'ParticipantID': pid, 'pool': 'test'})
    for pid in val_ids:
        pool_assignment.append({'ParticipantID': pid, 'pool': 'val'})
    for pid in train_ids:
        pool_assignment.append({'ParticipantID': pid, 'pool': 'train'})

    df_pools = pd.DataFrame(pool_assignment)
    df_pools.to_parquet(pool_file)
    print('Pool split created and saved permanently.')

section('Pool split summary')
print(df_pools['pool'].value_counts())
print(f'\nTotal participants assigned: {len(df_pools)}')

section('IMPORTANT — demo pool participant IDs (never used below this point)')
demo_list = df_pools[df_pools['pool'] == 'demo']['ParticipantID'].tolist()
for pid in demo_list:
    print(pid)

Loading participant-info (self-contained, no dependency on earlier cells)
Downloaded participant_info.csv.gz
Loaded 212 participants
Locking participant pool split
No existing split found — creating it now, ONE TIME ONLY.
Pool split created and saved permanently.
Pool split summary
pool
train    140
val       30
test      27
demo      15
Name: count, dtype: int64

Total participants assigned: 212
IMPORTANT — demo pool participant IDs (never used below this point)
e3e5e4aa-5950-4f1f-915c-c67598965b03
9c4b0b2a-92e1-4aeb-be90-98f65c1d4217
132c620a-fa7c-4f95-8dbf-9edc6f973336
02581754-36cd-4b23-85ea-bf995c6dec83
c7492565-48cd-4b52-af25-3aee61a391ad
1586a0ff-0e95-4b1d-a2bd-97863a02811b
e05bec91-93e2-4b23-9d4f-ae79dade8451
eb4e1be4-29de-4120-9727-0ce8041da479
8c0d11a7-1838-4db4-9319-5d171522655d
19097e81-ae0b-4ef1-b5aa-84350c2feeb9
0ec84778-1a98-4cd7-aa11-05997ddadd52
ba240e43-900d-4477-8718-b9487ed24d7d
e6081755-d7f3-4c26-ab40-0709b8ad41ef
4720363c-e587-4dcf-bc00-1b3bb42e7dc8
a7db3311-aa07-

# Cell 25
#### Audit the /audio folder before writing any feature extraction code
#### List structure, download README, and inspect one real file for the sample participant
#### This confirms format, duration, and whether extraction is even feasible in the time left

In [26]:


section('audio — folder listing')
ftp = connect_ftp()
ftp.cwd('/audio')
audio_listing = ftp.nlst()
ftp.quit()
print(audio_listing)

section('audio — README (if present at top level)')
try:
    audio_readme_path = download_if_missing('/audio/README.md', 'audio_README.md')
    with open(audio_readme_path, 'r', errors='ignore') as f:
        print(f.read())
except Exception as e:
    print(f'No top-level README, or error: {e}')
    print('Will need to check inside subfolders instead.')

audio — folder listing
['fg-predictions', 'fg-predictions-csv', 'raw-features']
audio — README (if present at top level)
Download failed for /audio/README.md: 550 No such file or directory.
No top-level README, or error: 550 No such file or directory.
Will need to check inside subfolders instead.


# Cell 25b
#### No top-level README — check inside each audio subfolder instead
#### Lists contents and downloads README for fg-predictions, fg-predictions-csv, raw-features

In [27]:


for sub in ['fg-predictions', 'fg-predictions-csv', 'raw-features']:
    section(f'audio/{sub} — listing')
    ftp = connect_ftp()
    ftp.cwd(f'/audio/{sub}')
    items = ftp.nlst()
    ftp.quit()
    print(f'{len(items)} items, sample: {items[:5]}')

    try:
        readme_path = download_if_missing(f'/audio/{sub}/README.md', f'audio_{sub}_README.md')
        with open(readme_path, 'r', errors='ignore') as f:
            print(f.read())
    except Exception as e:
        print(f'No README here either: {e}')

audio/fg-predictions — listing
181 items, sample: ['02581754-36cd-4b23-85ea-bf995c6dec83', '0271c478-a56a-4c09-ab91-9743184dd71b', '02b7a595-6508-46bd-8239-6deb433d6290', '05dedb61-63bc-44e3-8e28-a5d32d91f7e9', '06b33ec4-706d-462f-a681-05491be38eb3']
Downloaded audio_fg-predictions_README.md
# Audio `fg_predictions`

One file per file in the `raw-features` folder (one to one mapping of file name and path).
Contains a single column vector with as many entries as frame in the corresponding `raw-features` file.
Each entry gives the predicted foreground between 0 (extremely unlikely foreground speech from the participant) and 1 (very likely foreground speech).

audio/fg-predictions-csv — listing
181 items, sample: ['02581754-36cd-4b23-85ea-bf995c6dec83', '0271c478-a56a-4c09-ab91-9743184dd71b', '02b7a595-6508-46bd-8239-6deb433d6290', '05dedb61-63bc-44e3-8e28-a5d32d91f7e9', '06b33ec4-706d-462f-a681-05491be38eb3']
Downloaded audio_fg-predictions-csv_README.md
# Audio `fg_predictions`

One fil

# Cell 25c
#### Check real size/shape for ONE participant before deciding audio feature scope
#### Frames are 10-60ms — need to know if this is feasible in remaining time

In [28]:


section('Checking real file structure — audio raw-features and fg-predictions-csv')

ftp = connect_ftp()
ftp.cwd(f'/audio/raw-features/{SAMPLE_PID}')
raw_files = ftp.nlst()
ftp.quit()
print(f'raw-features/{SAMPLE_PID}: {len(raw_files)} files, sample: {raw_files[:5]}')

ftp = connect_ftp()
ftp.cwd(f'/audio/fg-predictions-csv/{SAMPLE_PID}')
fg_files = ftp.nlst()
ftp.quit()
print(f'fg-predictions-csv/{SAMPLE_PID}: {len(fg_files)} files, sample: {fg_files[:5]}')

if raw_files:
    one_raw = download_if_missing(
        f'/audio/raw-features/{SAMPLE_PID}/{raw_files[0]}',
        f'audio_raw_sample.csv.gz'
    )
    df_audio_sample = pd.read_csv(one_raw, compression='gzip')
    print(f'\nOne raw-features file: {df_audio_sample.shape[0]} rows, {df_audio_sample.shape[1]} cols')
    print(list(df_audio_sample.columns)[:10])

Checking real file structure — audio raw-features and fg-predictions-csv
raw-features/02b7a595-6508-46bd-8239-6deb433d6290: 7591 files, sample: ['1521170191785.csv.gz', '1521171045061.csv.gz', '1521172617281.csv.gz', '1521172751864.csv.gz', '1521172918441.csv.gz']
fg-predictions-csv/02b7a595-6508-46bd-8239-6deb433d6290: 7589 files, sample: ['1521170191785.csv.gz', '1521171045061.csv.gz', '1521172617281.csv.gz', '1521172751864.csv.gz', '1521172918441.csv.gz']
Downloaded audio_raw_sample.csv.gz

One raw-features file: 1970 rows, 34 cols
['frameIndex', 'frameTime', 'F0final_sma', 'voicingFinalUnclipped_sma', 'jitterLocal_sma', 'jitterDDP_sma', 'shimmerLocal_sma', 'logHNR_sma', 'voiceProb_sma', 'F0_sma']


# Cell 26
#### Full feature build: fitbit heart-rate + daily-summary + EMA context fields
#### Covers Objectives 1-3 (workload/behavioral, personalized baseline, temporal)
#### STRICTLY EXCLUDES demo pool — train/val/test participants only
#### Assumes df_pools, df_ema_resp, and demo_list already exist in memory
#### (re-run Cell 24 first if this session was restarted)


In [29]:

section('Building full feature table — heart-rate + daily-summary + EMA context')

modeling_ids = df_pools[df_pools['pool'].isin(['train', 'val', 'test'])]['ParticipantID'].tolist()
print(f'Participants included: {len(modeling_ids)} (demo pool of {len(demo_list)} excluded)')

all_rows = []
failed_ids = []

for i, pid in enumerate(modeling_ids):
    try:
        # --- Heart rate (already verified reliable) ---
        hr_p = download_if_missing(f'/fitbit/heart-rate/{pid}.csv.gz', f'hr_{pid}.csv.gz')
        df_hr_p = pd.read_csv(hr_p, compression='gzip')
        df_hr_p['Timestamp_parsed'] = pd.to_datetime(df_hr_p['Timestamp'], errors='coerce')
        df_hr_p['date'] = df_hr_p['Timestamp_parsed'].dt.date

        daily_hr = df_hr_p.groupby('date')['HeartRatePPG'].agg(['mean', 'std', 'min', 'max']).reset_index()
        daily_hr.columns = ['date', 'hr_mean', 'hr_std', 'hr_min', 'hr_max']

        personal_baseline = daily_hr['hr_mean'].mean()
        daily_hr['hr_mean_deviation'] = daily_hr['hr_mean'] - personal_baseline
        daily_hr = daily_hr.sort_values('date')
        daily_hr['hr_dev_roll3'] = daily_hr['hr_mean_deviation'].rolling(3, min_periods=1).mean()
        daily_hr['hr_dev_roll7'] = daily_hr['hr_mean_deviation'].rolling(7, min_periods=1).mean()

        # --- Daily summary: steps + activity minutes (workload proxy) ---
        ds_p = download_if_missing(f'/fitbit/daily-summary/{pid}.csv.gz', f'ds_{pid}.csv.gz')
        df_ds_p = pd.read_csv(ds_p, compression='gzip')
        df_ds_p['date'] = pd.to_datetime(df_ds_p['Timestamp'], errors='coerce').dt.date

        activity_cols = ['NumberSteps', 'Cardio_minutes', 'Fat Burn_minutes',
                          'Peak_minutes', 'Out of Range_minutes', 'RestingHeartRate']
        daily_ds = df_ds_p[['date'] + activity_cols].copy()

        steps_baseline = daily_ds['NumberSteps'].mean()
        daily_ds['steps_deviation'] = daily_ds['NumberSteps'] - steps_baseline

        # --- Merge heart-rate + daily-summary ---
        daily_combined = daily_hr.merge(daily_ds, on='date', how='outer')

        # --- EMA responses + context (activity/location at time of survey) ---
        ema_p = df_ema_resp[df_ema_resp['participant_id'] == pid].copy()
        ema_p['date'] = ema_p['sent_ts_parsed'].dt.date
        ema_cols = ['participant_id', 'date', 'stress', 'context2', 'context3']
        ema_cols = [c for c in ema_cols if c in ema_p.columns]

        merged = ema_p[ema_cols].merge(daily_combined, on='date', how='inner')
        all_rows.append(merged)

    except Exception as e:
        failed_ids.append((pid, str(e)))

    if (i + 1) % 50 == 0:
        print(f'Processed {i + 1}/{len(modeling_ids)} participants...')

section('Feature build complete')

df_features = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
print(f'Total rows: {len(df_features)}')
print(f'Participants processed: {len(all_rows)}, failed: {len(failed_ids)}')
if failed_ids:
    print('Failed (first 10):', failed_ids[:10])

section('Feature table preview')
print(df_features.head(10).to_string(index=False))

section('Saving feature table to cache')
feature_cache_path = f'{CACHE}/model_a_features_trainvaltest.parquet'
df_features.to_parquet(feature_cache_path)
print(f'Saved to: {feature_cache_path}')
print(f'Columns: {list(df_features.columns)}')

Building full feature table — heart-rate + daily-summary + EMA context
Participants included: 197 (demo pool of 15 excluded)
Downloaded hr_8f1c82eb-5e71-4187-a0a3-f2b58d8d99bd.csv.gz
Downloaded ds_8f1c82eb-5e71-4187-a0a3-f2b58d8d99bd.csv.gz
Downloaded hr_14121de2-f38e-4906-9fbe-b613549623fd.csv.gz
Downloaded ds_14121de2-f38e-4906-9fbe-b613549623fd.csv.gz
Downloaded hr_77dfe9b8-5f40-49a6-ba09-706198bb8a48.csv.gz
Downloaded ds_77dfe9b8-5f40-49a6-ba09-706198bb8a48.csv.gz
Downloaded hr_f610ffea-f6cb-4182-bbe7-19ff8fbe66ee.csv.gz
Downloaded ds_f610ffea-f6cb-4182-bbe7-19ff8fbe66ee.csv.gz
Downloaded hr_95f9f3a8-128c-4382-aa4b-6ea319b497ae.csv.gz
Downloaded ds_95f9f3a8-128c-4382-aa4b-6ea319b497ae.csv.gz
Downloaded hr_75c633d4-6ade-471e-983a-aadc07d9383c.csv.gz
Downloaded ds_75c633d4-6ade-471e-983a-aadc07d9383c.csv.gz
Downloaded hr_658adbe4-781c-45f9-92a7-14912fcd0701.csv.gz
Downloaded ds_658adbe4-781c-45f9-92a7-14912fcd0701.csv.gz
Downloaded hr_9548d1c4-bd4a-4841-9146-c0678135069d.csv.gz
Downl

# Cell 27 (fixed — correct column name: timeStamp, not timestamp)
#### Build the proximity/centrality feature from jelly data
#### Fixed: uses 'timeStamp' (capital S) matching the real file structure
#### Files are already downloaded from the previous run — this just reprocesses them
#### Merge into the Model A feature table, then drop rows with missing stress

In [32]:


section('Building jelly-based proximity/centrality feature — train/val/test only')

centrality_rows = []
jelly_failed = []

for i, pid in enumerate(modeling_ids):
    try:
        jelly_p = download_if_missing(f'/owlinone/jelly/{pid}.csv.gz', f'jelly_{pid}.csv.gz')
        df_jelly_p = pd.read_csv(jelly_p, compression='gzip')
        df_jelly_p['date'] = pd.to_datetime(df_jelly_p['timeStamp'], errors='coerce').dt.date

        daily_centrality = df_jelly_p.groupby('date').agg(
            distinct_receivers=('receiverDirectory', 'nunique'),
            total_detections=('receiverDirectory', 'count'),
        ).reset_index()
        daily_centrality['participant_id'] = pid

        centrality_rows.append(daily_centrality)

    except Exception as e:
        jelly_failed.append((pid, str(e)))

    if (i + 1) % 50 == 0:
        print(f'Processed {i + 1}/{len(modeling_ids)} participants for centrality...')

section('Centrality build complete')

df_centrality = pd.concat(centrality_rows, ignore_index=True) if centrality_rows else pd.DataFrame()
print(f'Total centrality rows: {len(df_centrality)}')
print(f'Participants with jelly data: {len(centrality_rows)}, failed: {len(jelly_failed)}')
if jelly_failed:
    print('Failed (first 10):', jelly_failed[:10])

section('Merging centrality into Model A/B combined feature table')

df_features_full = df_features.merge(
    df_centrality, on=['participant_id', 'date'], how='left'
)

section('Dropping rows with missing stress (unanswered EMA sends)')
before = len(df_features_full)
df_features_full = df_features_full.dropna(subset=['stress'])
after = len(df_features_full)
print(f'Rows before: {before}, after dropping missing stress: {after}')

section('Final combined feature table preview')
print(df_features_full.head(10).to_string(index=False))
print(f'\nRows with centrality data present: {df_features_full["distinct_receivers"].notna().sum()}')
print(f'Rows with centrality data missing: {df_features_full["distinct_receivers"].isna().sum()}')

section('Saving final combined feature table')
final_path = f'{CACHE}/model_ab_features_final.parquet'
df_features_full.to_parquet(final_path)
print(f'Saved to: {final_path}')
print(f'Final columns: {list(df_features_full.columns)}')

Building jelly-based proximity/centrality feature — train/val/test only
Download failed for /owlinone/jelly/8f1c82eb-5e71-4187-a0a3-f2b58d8d99bd.csv.gz: 550 No such file or directory.
Already have jelly_14121de2-f38e-4906-9fbe-b613549623fd.csv.gz, skipping download.
Already have jelly_77dfe9b8-5f40-49a6-ba09-706198bb8a48.csv.gz, skipping download.
Already have jelly_f610ffea-f6cb-4182-bbe7-19ff8fbe66ee.csv.gz, skipping download.
Already have jelly_95f9f3a8-128c-4382-aa4b-6ea319b497ae.csv.gz, skipping download.
Already have jelly_75c633d4-6ade-471e-983a-aadc07d9383c.csv.gz, skipping download.
Already have jelly_658adbe4-781c-45f9-92a7-14912fcd0701.csv.gz, skipping download.
Already have jelly_9548d1c4-bd4a-4841-9146-c0678135069d.csv.gz, skipping download.
Already have jelly_a9dfbe4d-4076-48c7-a72b-342fe4c12514.csv.gz, skipping download.
Already have jelly_16812063-e5df-4657-b86c-5b55a0c9ffe6.csv.gz, skipping download.
Already have jelly_0271c478-a56a-4c09-ab91-9743184dd71b.csv.gz, skipp

# Cell 28
#### Build the REAL person-to-person proximity graph from jelly data
#### Step 1: for each participant, in each 5-min window, keep the receiver
####         with strongest RSSI as their located room for that window
#### Step 2: combine all participants, group by date+window+room
#### Step 3: within each group, create an edge between every pair present
#### Step 4: degree centrality = distinct co-located people per day
####         weighted centrality = total co-location edge-instances per day
#### Uses files already downloaded — no new network calls

In [33]:


from itertools import combinations

section('Loading and reducing all jelly files to per-window located room')

located_rows = []
load_failed = []

for i, pid in enumerate(modeling_ids):
    local_path = f'{RAW}/jelly_{pid}.csv.gz'
    if not os.path.exists(local_path):
        load_failed.append(pid)
        continue
    try:
        df_j = pd.read_csv(local_path, compression='gzip')
        df_j['ts_parsed'] = pd.to_datetime(df_j['timeStamp'], errors='coerce')
        df_j['date'] = df_j['ts_parsed'].dt.date
        df_j['time_window'] = df_j['ts_parsed'].dt.floor('5min')

        # Keep the strongest-signal receiver per participant per time window
        idx = df_j.groupby('time_window')['rssi'].idxmax()
        located = df_j.loc[idx, ['date', 'time_window', 'receiverDirectory']].copy()
        located['participant_id'] = pid

        located_rows.append(located)

    except Exception as e:
        load_failed.append((pid, str(e)))

    if (i + 1) % 50 == 0:
        print(f'Processed {i + 1}/{len(modeling_ids)} participants...')

section('Combining all participants into one located-position table')

df_located = pd.concat(located_rows, ignore_index=True)
print(f'Total located rows: {len(df_located)}')
print(f'Participants included: {len(located_rows)}, failed: {len(load_failed)}')

section('Finding co-location groups and building pairwise edges')

edge_records = []
group_count = 0

for (date_val, window_val, receiver_val), grp in df_located.groupby(['date', 'time_window', 'receiverDirectory']):
    pids = grp['participant_id'].unique().tolist()
    if len(pids) < 2:
        continue
    group_count += 1
    for p1, p2 in combinations(pids, 2):
        edge_records.append({'date': date_val, 'p1': p1, 'p2': p2})

print(f'Co-location groups found (2+ people): {group_count}')
print(f'Pairwise edges created: {len(edge_records)}')

df_edges = pd.DataFrame(edge_records)

section('Computing daily degree and weighted centrality per participant')

# Make edges symmetric (both directions) so each participant sees all their contacts
df_edges_sym = pd.concat([
    df_edges.rename(columns={'p1': 'participant_id', 'p2': 'other'}),
    df_edges.rename(columns={'p2': 'participant_id', 'p1': 'other'}),
], ignore_index=True)

df_real_centrality = df_edges_sym.groupby(['participant_id', 'date']).agg(
    degree_centrality=('other', 'nunique'),
    weighted_centrality=('other', 'count'),
).reset_index()

print(f'Real centrality rows: {len(df_real_centrality)}')
print(df_real_centrality.head(10).to_string(index=False))

section('Saving real centrality table')
centrality_path = f'{CACHE}/real_centrality_daily.parquet'
df_real_centrality.to_parquet(centrality_path)
print(f'Saved to: {centrality_path}')

Loading and reducing all jelly files to per-window located room
Processed 50/197 participants...
Processed 100/197 participants...
Combining all participants into one located-position table
Total located rows: 391632
Participants included: 190, failed: 7
Finding co-location groups and building pairwise edges
Co-location groups found (2+ people): 29991
Pairwise edges created: 37850
Computing daily degree and weighted centrality per participant
Real centrality rows: 3712
                      participant_id       date  degree_centrality  weighted_centrality
0271c478-a56a-4c09-ab91-9743184dd71b 2018-05-05                  4                   31
0271c478-a56a-4c09-ab91-9743184dd71b 2018-05-07                  1                    2
0271c478-a56a-4c09-ab91-9743184dd71b 2018-05-18                  1                    1
0271c478-a56a-4c09-ab91-9743184dd71b 2018-06-04                  3                    4
0271c478-a56a-4c09-ab91-9743184dd71b 2018-06-06                  3                    

# Cell 29
#### Merge the REAL person-to-person centrality into the final feature table
#### Keeps the old proxy columns (distinct_receivers, total_detections) for
#### comparison/documentation, adds the real degree/weighted centrality
#### Reports final coverage — how many rows actually have real network data

In [34]:


section('Merging real centrality into final feature table')

df_model_ready = df_features_full.merge(
    df_real_centrality, on=['participant_id', 'date'], how='left'
)

section('Coverage summary')
total_rows = len(df_model_ready)
real_centrality_present = df_model_ready['degree_centrality'].notna().sum()
proxy_present = df_model_ready['distinct_receivers'].notna().sum()

print(f'Total rows in modeling table: {total_rows}')
print(f'Rows with REAL person-to-person centrality: {real_centrality_present} '
      f'({real_centrality_present/total_rows*100:.1f}%)')
print(f'Rows with proxy room-count centrality: {proxy_present} '
      f'({proxy_present/total_rows*100:.1f}%)')

section('Final modeling table preview')
preview_cols = ['participant_id', 'date', 'stress', 'hr_mean_deviation',
                'hr_dev_roll7', 'NumberSteps', 'degree_centrality', 'weighted_centrality']
print(df_model_ready[preview_cols].dropna(subset=['degree_centrality']).head(10).to_string(index=False))

section('Saving final modeling-ready table')
final_model_path = f'{CACHE}/model_ready_final.parquet'
df_model_ready.to_parquet(final_model_path)
print(f'Saved to: {final_model_path}')
print(f'Total columns: {len(df_model_ready.columns)}')
print(f'Column list: {list(df_model_ready.columns)}')

Merging real centrality into final feature table
Coverage summary
Total rows in modeling table: 10038
Rows with REAL person-to-person centrality: 2989 (29.8%)
Rows with proxy room-count centrality: 3841 (38.3%)
Final modeling table preview
                      participant_id       date  stress  hr_mean_deviation  hr_dev_roll7  NumberSteps  degree_centrality  weighted_centrality
14121de2-f38e-4906-9fbe-b613549623fd 2018-03-07     1.0          -3.384741     -1.831167        15209                5.0                 22.0
14121de2-f38e-4906-9fbe-b613549623fd 2018-03-08     2.0          -0.266020     -1.439880        14869                3.0                 12.0
14121de2-f38e-4906-9fbe-b613549623fd 2018-03-09     3.0          -2.505505     -1.653005        13574                2.0                  4.0
14121de2-f38e-4906-9fbe-b613549623fd 2018-03-12     3.0           1.046301     -1.739477        13942                3.0                 42.0
14121de2-f38e-4906-9fbe-b613549623fd 2018-03-13   

# Cell 30 (fixed — column name mismatch: ParticipantID vs participant_id)
#### Prepare final train/val/test splits for Model A vs Model B
#### Target: binary elevated-stress (stress >= 3) vs not, given class imbalance
#### Model A and Model B use the SAME matched subset (centrality present)
#### for a fair ablation comparison; full-sample Model A kept separately

In [36]:


section('Defining binary target and pool assignment')

df_model_ready['stress_binary'] = (df_model_ready['stress'] >= 3).astype(int)
print('Target distribution (full table):')
print(df_model_ready['stress_binary'].value_counts(normalize=True))

# Fix: df_pools uses 'ParticipantID', df_model_ready uses 'participant_id'
df_pools_renamed = df_pools.rename(columns={'ParticipantID': 'participant_id'})
df_model_ready = df_model_ready.merge(df_pools_renamed, on='participant_id', how='left')
print('\nPool distribution in modeling table:')
print(df_model_ready['pool'].value_counts())

section('Matched subset (centrality present) for the core A vs B ablation')

df_matched = df_model_ready.dropna(subset=['degree_centrality']).copy()
print(f'Matched subset size: {len(df_matched)}')
print('Target distribution (matched subset):')
print(df_matched['stress_binary'].value_counts(normalize=True))
print('Pool distribution (matched subset):')
print(df_matched['pool'].value_counts())

section('Defining feature sets')

features_a = ['hr_mean', 'hr_std', 'hr_min', 'hr_max', 'hr_mean_deviation',
              'hr_dev_roll3', 'hr_dev_roll7', 'NumberSteps', 'Cardio_minutes',
              'Fat Burn_minutes', 'Peak_minutes', 'Out of Range_minutes',
              'RestingHeartRate', 'steps_deviation']

features_b = features_a + ['degree_centrality', 'weighted_centrality']

print(f'Model A features ({len(features_a)}): {features_a}')
print(f'Model B features ({len(features_b)}): {features_b}')

section('Splitting matched subset into train/val/test by pool')

train_matched = df_matched[df_matched['pool'] == 'train']
val_matched = df_matched[df_matched['pool'] == 'val']
test_matched = df_matched[df_matched['pool'] == 'test']

print(f'Train rows: {len(train_matched)}')
print(f'Val rows: {len(val_matched)}')
print(f'Test rows: {len(test_matched)}')

section('Saving prepared splits')
train_matched.to_parquet(f'{CACHE}/split_train_matched.parquet')
val_matched.to_parquet(f'{CACHE}/split_val_matched.parquet')
test_matched.to_parquet(f'{CACHE}/split_test_matched.parquet')
print('Saved train/val/test matched splits to cache.')

Defining binary target and pool assignment
Target distribution (full table):
stress_binary
0    0.754334
1    0.245666
Name: proportion, dtype: float64

Pool distribution in modeling table:
pool
train    6883
test     1612
val      1543
Name: count, dtype: int64
Matched subset (centrality present) for the core A vs B ablation
Matched subset size: 2989
Target distribution (matched subset):
stress_binary
0    0.693878
1    0.306122
Name: proportion, dtype: float64
Pool distribution (matched subset):
pool
train    2018
val       500
test      471
Name: count, dtype: int64
Defining feature sets
Model A features (14): ['hr_mean', 'hr_std', 'hr_min', 'hr_max', 'hr_mean_deviation', 'hr_dev_roll3', 'hr_dev_roll7', 'NumberSteps', 'Cardio_minutes', 'Fat Burn_minutes', 'Peak_minutes', 'Out of Range_minutes', 'RestingHeartRate', 'steps_deviation']
Model B features (16): ['hr_mean', 'hr_std', 'hr_min', 'hr_max', 'hr_mean_deviation', 'hr_dev_roll3', 'hr_dev_roll7', 'NumberSteps', 'Cardio_minutes', '